# 평가셋(Evaluation DataSet) 구축

### 환경 설정

In [11]:
import os
import sqlite3
import pandas as pd
import chromadb

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain.agents import create_agent
from pathlib import Path
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer


In [12]:
load_dotenv('.env')
load_dotenv('/.env')

if not os.getenv('OPENAI_API_KEY') :
    raise RuntimeError(
        'OpenAI API Key를 확인해 보세요.'
    )
else :
    model = ChatOpenAI(model = 'gpt-4o-mini', temperature=1, timeout=60)
    print('OpenAI API 연결 & MODEL 생성 완료')

OpenAI API 연결 & MODEL 생성 완료


### VectorDB (ChromaDB) 연결 / 임베딩 모델 생성

In [13]:
DB_PATH = Path("../chroma_db")
COLLECTION_NAME = "maplestory_guides"

#_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, isolation_level=None, check_same_thread=False)
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_collection(name=COLLECTION_NAME)

print('데이터베이스 연결 및 임베딩 모델 생성 완료')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3213.06it/s]


데이터베이스 연결 및 임베딩 모델 생성 완료


### 평가셋 TEST

#### 평가셋 데이터 가져오기

In [14]:
# [rag_eval_questions_100.csv](/C:/Users/Playdata/Desktop/mle-01-p1-team3/docs/rag_eval_questions_100.csv)

evalset = pd.read_csv('../docs/rag_eval_gold_100.csv')
print('페이지 수 : ', len(evalset))

display(evalset.head(3))


페이지 수 :  100


,eval_id,persona,category,question,question_variant,source_type,difficulty,answerable,gold_source_refs,gold_source_ids,gold_source_ref_ids,gold_source_urls,reference_answer,must_include,nice_to_include,must_not_include,scoring_focus
0,E001,초보,게임시작,메이플스토리를 처음 시작하려면 무엇부터 해야 하나요?,처음 설치한 뒤 어떤 순서로 시작하면 되나요?,guide,easy,yes,게임 시작,chunk_0; chunk_1; chunk_2; chunk_3; chunk_4; c...,article:272,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"
1,E002,초보,게임시작,메이플스토리는 어떻게 설치하고 실행하나요?,게임을 시작하려면 설치 절차가 어떻게 되나요?,guide,easy,yes,게임 시작,chunk_0; chunk_1; chunk_2; chunk_3; chunk_4; c...,article:272,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"
2,E003,초보,게임시작,처음 시작할 때 선택할 수 있는 직업은 어떤 종류가 있나요?,초반에 어떤 직업군을 고를 수 있나요?,guide,easy,yes,메이플스토리 신규 · 복귀 용사 가이드,chunk_1378; chunk_1379; chunk_1380; chunk_1381...,article:147377,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"


##### 함수 정의

In [15]:
def search_ids(question: str, k: int = 3):
    """질문과 가장 가까운 조각 k개의 id를 순위 순서로 돌려준다."""
    query_embedding = embed_model.encode(
        [question],
        normalize_embeddings = True
    )

    result = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results = k,
        include = []
    )

    return result["ids"][0]

In [16]:
# 평가셋(evalset)구성
# 'eval_id', 'persona', 'category', 'question', 
# 'question_variant', 'source_type', 'difficulty', 
# 'answerable', 'gold_source_refs', 'gold_source_ids', 
# 'gold_source_ref_ids', 'gold_source_urls', 'reference_answer', 
# 'must_include', 'nice_to_include', 'must_not_include', 
# 'scoring_focus'

# 가장 무난한 질문 선택 (또는 평가셋 번호, 예: 'E001' ~ 'E100')
credit_eval_id = "E003"

# 1. 평가셋에서 문항 찾기
credit_row = evalset[evalset["eval_id"] == credit_eval_id].iloc[0]
credit_query = credit_row["question"]

# 2. 실제 질문으로 검색
K = 3
top3 = search_ids(credit_query, k=K)

for rank, chunk_id in enumerate(top3, 1):
    print(f"{rank}위 {chunk_id}")

# 3. gold chunk_id 정리
credit_gold = [x.strip() for x in credit_row["gold_source_ids"].split(";") if x.strip()]

print(f"평가셋 문항 {credit_row['eval_id']}의 정답 조각 : {credit_gold}")

print("=" * 100)

# 4. chunk 단위 평가
hits = len(set(top3) & set(credit_gold))   # 상위 K개 중 정답 개수

hit_k = int(bool(hits))
recall_k = hits / len(set(credit_gold))    # 분모는 그 문항의 정답 개수
precision_k = hits / K                     # 분모는 언제나 K
# Precision 의 분모를 len(top3) 로 두면, 검색이 K개보다 적게 돌려줄수록
# 점수가 좋아지는 이상한 지표가 된다. 같은 K 끼리 비교하려면 K 로 고정한다.

print("Hit@K 평가 :", hit_k)
print("Recall@K 평가 :", recall_k)
print("Precision@K 평가 :", precision_k)

1위 chunk_1380
2위 chunk_1390
3위 chunk_7
평가셋 문항 E003의 정답 조각 : ['chunk_1378', 'chunk_1379', 'chunk_1380', 'chunk_1381', 'chunk_1382', 'chunk_1383', 'chunk_1384', 'chunk_1385', 'chunk_1386', 'chunk_1387', 'chunk_1388', 'chunk_1389', 'chunk_1390', 'chunk_1391', 'chunk_1392', 'chunk_1393', 'chunk_1394', 'chunk_1395', 'chunk_1396', 'chunk_1397', 'chunk_1398', 'chunk_1399', 'chunk_1400', 'chunk_1401', 'chunk_1402', 'chunk_1403', 'chunk_1404', 'chunk_1405', 'chunk_1406', 'chunk_1407', 'chunk_1408', 'chunk_1409', 'chunk_1410', 'chunk_1411', 'chunk_1412', 'chunk_1413', 'chunk_1414', 'chunk_1415', 'chunk_1416', 'chunk_1417', 'chunk_1418', 'chunk_1419', 'chunk_1420', 'chunk_1421', 'chunk_1422', 'chunk_1423', 'chunk_1424', 'chunk_1425', 'chunk_1426', 'chunk_1427', 'chunk_1428', 'chunk_1429', 'chunk_1430', 'chunk_1431', 'chunk_1432', 'chunk_1433', 'chunk_1434', 'chunk_1435', 'chunk_1436', 'chunk_1437', 'chunk_1438', 'chunk_1439', 'chunk_1440']
Hit@K 평가 : 1
Recall@K 평가 : 0.031746031746031744
Precision

In [17]:
print('질문 : ', credit_query)
print()

for rank in top3 :
    result = collection.get(ids=rank, include=["documents", "metadatas"])
    print(result["documents"][0])
    print(result["metadatas"][0])
    print()

for gold in credit_gold :
    result = collection.get(ids=gold, include=["documents", "metadatas"])
    print(result["documents"][0])
    print(result["metadatas"][0])
    print()


질문 :  처음 시작할 때 선택할 수 있는 직업은 어떤 종류가 있나요?

클릭하면 자세한 내용을 확인하실 수 있습니다 START 게임 시작하기 Q메이플스토리는 어떻게 시작하나요 Q어떤 직업을 선택할 수 있나요 Q처음 시작하면 무엇부터 해야 하나요 RETURN 복귀 용사님을 위한 안내 Q복귀하면 기존 캐릭터를 이어서 키울 수 있나요 Q오랜만에 메이플스토리에 돌아왔는데 바뀐 시스템에 잘 적응할 수 있을까요 GROWTH 캐릭터 성장
{'chunk_index': 1380, 'name': '메이플스토리 신규 · 복귀 용사 가이드', 'source': 'guide', 'section_title': '기타/TIP', 'article_id': 147377, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/147377', 'board_id': 429467345}

Q-어떤 직업을 선택할 수 있나요? A-메이플스토리에는 전사 · 마법사 · 궁수 · 도적 · 해적의 다섯 계열로 구분되는 여러 다양한 직업이 존재하며, 출신에 따라 모험가 · 시그너스 기사단 · 영웅 · 레지스탕스 등의 직업군으로 나뉩니다. 직업마다 전투 방식과 조작법, 고유한 스토리가 달라 용사님의 플레이 스타일에 맞는 직업을 선택하실 수 있습니다.
{'chunk_index': 1390, 'board_id': 429467345, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/147377', 'name': '메이플스토리 신규 · 복귀 용사 가이드', 'section_title': '기타/TIP', 'source': 'guide', 'article_id': 147377}

목차 1 캐릭터 직업 선택하기 2 캐릭터 설정하기 3 캐릭터 삭제하기 캐릭터 생성 버튼을 클릭하여 캐릭터를 생성할 수 있습니다 한 월드당 기본 12개의 캐릭터를 만들 수 있으며 챌린저스 월드 

### 전체 평가 - 네 가지 지표

한 문항만 보던 것을 평가셋 전체로 넓힙니다. `search_ids` 로 검색하고, 아래 네 함수로 잽니다.

> **읽을 때 주의** - 이 평가셋의 `gold_source_ids` 는 대부분 article 전체 청크가 들어 있어
> 문항당 정답이 평균 21개입니다. 정답 개수가 K 보다 많으면 Recall 은 1.0 이 될 수 없으므로,
> **Recall 은 반드시 `정답수` 열과 함께** 읽습니다.

In [18]:
# 네 지표를 직접 구현합니다 -- 이 함수들로 평가셋 전체를 잽니다.
#  predicted -> 검색이 돌려준 조각 id 목록(순위 순), relevant -> 그 질문의 정답 조각 id 목록.
def hit_at_k(predicted, relevant, k):
    """상위 k개 중 정답이 하나라도 있으면 1, 없으면 0."""
    return 1 if any(p in relevant for p in predicted[:k]) else 0


def precision_at_k(predicted, relevant, k):
    """상위 k개 중 정답의 비율(정답 개수 / k)."""
    # 나누는 수는 언제나 k 다 -- 검색 결과가 k 개보다 적게 나와도 k 로 나눈다.
    return sum(1 for p in predicted[:k] if p in relevant) / k


def recall_at_k(predicted, relevant, k):
    """전체 정답 중 상위 k개가 건진 비율(찾은 정답 수 / 전체 정답 수)."""
    # 세는 것은 Precision 과 같고 나누는 수만 다르다 -- 여기는 그 질문의 정답 개수.
    #  그래서 정답 개수가 k 보다 많으면 이 값은 1.0 이 될 수가 없다.
    return sum(1 for p in predicted[:k] if p in relevant) / len(relevant)


def mrr_at_k(predicted, relevant, k):
    """첫 정답 순위의 역수(1위면 1, 2위면 0.5 ...). 상위 k개 안에 없으면 0."""
    # enumerate 의 두 번째 인자 1 은 순위를 0 이 아니라 1 부터 세게 한다.
    for rank, p in enumerate(predicted[:k], 1):
        if p in relevant:
            # 처음 만난 정답에서 바로 끝낸다 -- MRR 이 보는 것은 '첫' 정답의 순위뿐이다.
            return 1 / rank
    return 0.0

In [19]:
# 라벨을 '문항 id -> 정답 조각 목록' 사전으로 만들어 둡니다.
#  gold_source_ids 는 CSV 한 칸에 담느라 ';' 로 이어 붙인 문자열이라, 견주려면 목록으로 되돌려야 합니다.
def parse_gold(cell):
    if pd.isna(cell):
        return []
    return [chunk.strip() for chunk in str(cell).split(";") if chunk.strip()]


evalset["gold"] = evalset["gold_source_ids"].apply(parse_gold)
evalset["정답수"] = evalset["gold"].apply(len)

# gold 가 비어 있는 문항(answerable=no)은 Recall 의 분모가 0 이라 검색 지표로 잴 수 없습니다.
scorable = evalset[evalset["정답수"] > 0].copy()
print(f"제외한 문항: {len(evalset) - len(scorable)}개 (gold 없음)")

gold_map = {row.eval_id: row.gold for row in scorable.itertuples()}

제외한 문항: 5개 (gold 없음)


In [20]:
def evaluate(k):
    """문항마다 네 지표를 재서 DataFrame 으로 돌려준다."""
    rows = []
    for row in scorable.itertuples():
        # 검색은 문항마다 한 번만 합니다 -- 지표가 네 개라고 네 번 검색할 이유가 없습니다.
        predicted = search_ids(row.question, k)
        relevant = gold_map[row.eval_id]
        # 네 지표 모두 위에서 만든 함수로 냅니다 -- 검색 결과와 정답 목록만 있으면 됩니다.
        rows.append({"eval_id": row.eval_id,
                     "Hit": hit_at_k(predicted, relevant, k),
                     "P": precision_at_k(predicted, relevant, k),
                     "R": recall_at_k(predicted, relevant, k),
                     "MRR": mrr_at_k(predicted, relevant, k)})
    return pd.DataFrame(rows)


scores = evaluate(3)
print("잰 문항 수:", len(scores))

잰 문항 수: 95


In [21]:
# 평균보다 '질문별 표' 를 먼저 봅니다 -- 어느 질문이 틀렸는지가 평균보다 먼저 알아야 할 것입니다.
detail = scores.merge(scorable[["eval_id", "question", "정답수"]], on="eval_id")
display(detail[["eval_id", "Hit", "P", "R", "MRR", "정답수", "question"]].round(3))

,eval_id,Hit,P,R,MRR,정답수,question
0,E001,0,0.000,0.000,0.0,6,메이플스토리를 처음 시작하려면 무엇부터 해야 하나요?
1,E002,1,0.333,0.167,0.5,6,메이플스토리는 어떻게 설치하고 실행하나요?
2,E003,1,0.667,0.032,1.0,63,처음 시작할 때 선택할 수 있는 직업은 어떤 종류가 있나요?
3,E004,1,0.667,0.032,1.0,63,친구와 함께 시작하려면 무엇을 맞춰야 하나요?
4,E005,0,0.000,0.000,0.0,10,캐릭터는 어떻게 생성하나요?
...,...,...,...,...,...,...,...
90,E091,1,0.667,0.020,1.0,98,하루에 오래 못 하는 초보는 무엇부터 하면 좋나요?
91,E092,1,0.667,0.041,1.0,49,보스 입문 전에 기본적으로 챙겨야 할 것은 무엇인가요?
92,E093,1,0.667,0.061,0.5,33,장비를 처음 맞출 때 강화 순서를 어떻게 잡으면 좋나요?
93,E094,1,1.000,0.031,1.0,96,복귀 유저가 다시 적응하려면 어떤 순서로 시스템을 보면 좋을까요?


In [22]:
# 그다음이 평균입니다. Hit 의 평균은 '정답을 하나라도 찾은 문항의 비율' 입니다.
print("K = 3 평균")
print(scores[["Hit", "P", "R", "MRR"]].mean().round(3).to_string())
print()
print(f"Hit  : {int(scores['Hit'].sum())} / {len(scores)} 문항")

# 평균 뒤에 가려진 실패 문항을 꺼내 봅니다.
failed = detail[detail["Hit"] == 0]
print()
print(f"상위 3개에서 정답을 못 찾은 문항 {len(failed)}개:", failed["eval_id"].tolist())

K = 3 평균
Hit    0.832
P      0.582
R      0.218
MRR    0.751

Hit  : 79 / 95 문항

상위 3개에서 정답을 못 찾은 문항 16개: ['E001', 'E005', 'E011', 'E025', 'E032', 'E035', 'E036', 'E046', 'E052', 'E062', 'E073', 'E076', 'E080', 'E081', 'E084', 'E085']
